# BTL02 - FER2013 Model Training on Google Colab (GPU T4)
Notebook này dùng để huấn luyện tự động 3 mô hình Custom CNN (SmallCNN, MediumCNN, LargeCNN) trên GPU miễn phí của Google Colab.
**LƯU Ý QUAN TRỌNG:** Hãy vào menu **Runtime** -> **Change runtime type** -> chọn **T4 GPU** trước khi chạy!


In [ ]:
# 1. Kéo mã nguồn dự án từ nhánh của nhóm về Colab
!git clone -b feat/BTL02_T0103_T0104_T0105_T0201_T0202_data_and_cnn_models https://github.com/Duy-DS/Deep_Learning_Group.git
%cd Deep_Learning_Group/BTL02


In [ ]:
# 2. Cài đặt các thư viện cần thiết cho MLOps
!pip install -q mlflow pyyaml


In [ ]:
# 3. Tải và giải nén bộ dữ liệu chuẩn fer2013.csv (~96MB nén)
import urllib.request
import zipfile
import os

os.makedirs('data/raw', exist_ok=True)
csv_target = 'data/raw/fer2013.csv'
zip_target = 'data/raw/fer2013.csv.zip'
dataset_url = 'https://huggingface.co/datasets/chitradrishti/fer2013/resolve/main/fer2013.csv.zip'

if not os.path.exists(csv_target):
    print('Đang tải dữ liệu fer2013.csv.zip...')
    urllib.request.urlretrieve(dataset_url, zip_target)
    print('Đang giải nén...')
    with zipfile.ZipFile(zip_target, 'r') as z:
        z.extractall('data/raw')
    if os.path.exists(zip_target):
        os.remove(zip_target)
    print('Tải và giải nén hoàn tất! Kích thước file:', os.path.getsize(csv_target), 'bytes')
else:
    print('File dữ liệu đã tồn tại sẵn.')


In [ ]:
# 4. Kiểm tra GPU
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))


In [ ]:
# 5. Huấn luyện Baseline SmallCNN (EXP-01 - 2 blocks Conv)
from src.train.trainer import Trainer

print('=' * 60)
print('🚀 BẮT ĐẦU HUẤN LUYỆN EXP-01 (SmallCNN)...')
print('=' * 60)
trainer_small = Trainer(config='experiments/baseline/exp01_small_cnn.yaml')
res_small = trainer_small.fit()


In [ ]:
# 6. Huấn luyện MediumCNN (EXP-02 - 3 blocks Conv)
print('=' * 60)
print('🚀 BẮT ĐẦU HUẤN LUYỆN EXP-02 (MediumCNN)...')
print('=' * 60)
trainer_medium = Trainer(config='experiments/baseline/exp02_medium_cnn.yaml')
res_medium = trainer_medium.fit()


In [ ]:
# 7. Huấn luyện LargeCNN (EXP-03 - 4 blocks + BatchNorm + Dropout)
print('=' * 60)
print('🚀 BẮT ĐẦU HUẤN LUYỆN EXP-03 (LargeCNN)...')
print('=' * 60)
trainer_large = Trainer(config='experiments/baseline/exp03_large_cnn.yaml')
res_large = trainer_large.fit()


In [ ]:
# 8. Xem bảng tổng hợp 24 cột MLflow và tải về máy tính
import pandas as pd
from google.colab import files

summary_df = pd.read_csv('outputs/logs/mlflow_summary_24col.csv')
print('=== TỔNG HỢP CẢ 3 THỰC NGHIỆM TRONG MLFLOW ===')
display(summary_df[['Run ID', 'params.architecture', 'metrics.best_val_acc', 'metrics.private_test_acc', 'metrics.macro_f1', 'metrics.best_epoch']])

# Tải file CSV tổng hợp về máy để dán vào Excel
files.download('outputs/logs/mlflow_summary_24col.csv')

# Tải các checkpoints tốt nhất về máy nếu cần lưu trữ
files.download('outputs/checkpoints/best_small_cnn.pth')
files.download('outputs/checkpoints/best_medium_cnn.pth')
files.download('outputs/checkpoints/best_large_cnn.pth')
